In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
file_path = "/Volumes/ecommerce_lakehouse/raw/oltp_landing/products/olist_products_dataset.csv"

columns_names = spark.read \
                .format("csv") \
                .option("header","True") \
                .load(file_path) \
                .limit(5)


display(columns_names)


In [0]:
products_Schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", StringType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", DoubleType(), True),
    StructField("product_length_cm", DoubleType(), True),
    StructField("product_height_cm", DoubleType(), True),
    StructField("product_width_cm", DoubleType(), True)

])

In [0]:
products_stream_df =spark.readStream \
                .format("cloudFiles") \
                .option("cloudFiles.format", "csv") \
                .option("header", "True") \
                .schema(products_Schema) \
                .load("/Volumes/ecommerce_lakehouse/raw/oltp_landing/products/")

In [0]:
bronze_products_df = products_stream_df \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "load_date",
        current_date()
    ) \
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

In [0]:
query = (
    bronze_products_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/ecommerce_lakehouse/raw/checkpoints/products/"
        )
        .trigger(availableNow=True)
        .toTable(
            "ecommerce_lakehouse.bronze.products_raw"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.bronze.products_raw
LIMIT 10;